# Linear Regression Worksheet

This worksheet walks through simple and multiple linear regression from scratch and with `scikit-learn`.

Topics covered:
- Simple linear regression (least squares, by hand)
- Multiple linear regression (normal equations)
- Ordinary Least Squares with `scikit-learn`
- Model evaluation (RMSE, R²)
- Residual diagnostics
- Regularization (ridge / LASSO preview)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# scikit-learn
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.figsize"] = (10, 6)

## 1. Generate synthetic data

We create a dataset where `y` is a linear function of `x` plus Gaussian noise.

In [ ]:
np.random.seed(42)

n_samples = 100
X = np.linspace(0, 10, n_samples).reshape(-1, 1)
true_slope = 2.5
true_intercept = 1.0
noise = np.random.normal(0, 3.0, size=n_samples)
y = true_intercept + true_slope * X.ravel() + noise

print(f"True intercept: {true_intercept}")
print(f"True slope:     {true_slope}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
plt.scatter(X, y, alpha=0.7, label="Observed data")
plt.plot(X, true_intercept + true_slope * X.ravel(), color="red", linewidth=2, label="True line")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Synthetic Linear Data")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Simple linear regression by hand (OLS)

For a single predictor, the least-squares estimates are:

\(\hat{\beta}_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}\),
\(\hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}\)

In [ ]:
x = X.ravel()
x_mean = x.mean()
y_mean = y.mean()

beta_1 = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)
beta_0 = y_mean - beta_1 * x_mean

print(f"Hand-calculated intercept: {beta_0:.4f}")
print(f"Hand-calculated slope:     {beta_1:.4f}")

In [ ]:
y_hat_hand = beta_0 + beta_1 * x

plt.scatter(X, y, alpha=0.7, label="Observed data")
plt.plot(X, y_hat_hand, color="green", linewidth=2, label=f"Hand OLS: y = {beta_0:.2f} + {beta_1:.2f}x")
plt.plot(X, true_intercept + true_slope * X.ravel(), color="red", linewidth=2, linestyle="--", label="True line")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Simple Linear Regression - Hand OLS")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Simple linear regression with `scikit-learn`

In [ ]:
reg = LinearRegression()
reg.fit(X, y)

print(f"scikit-learn intercept: {reg.intercept_:.4f}")
print(f"scikit-learn slope:     {reg.coef_[0]:.4f}")

y_hat_sklearn = reg.predict(X)

rmse = np.sqrt(mean_squared_error(y, y_hat_sklearn))
r2 = r2_score(y, y_hat_sklearn)
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

In [ ]:
plt.scatter(X, y, alpha=0.7, label="Observed data")
plt.plot(X, y_hat_sklearn, color="blue", linewidth=2, label=f"sklearn OLS: y = {reg.intercept_:.2f} + {reg.coef_[0]:.2f}x")
plt.plot(X, true_intercept + true_slope * X.ravel(), color="red", linewidth=2, linestyle="--", label="True line")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Simple Linear Regression - scikit-learn")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Multiple linear regression

Generate data with multiple predictors and fit with the normal equation and `scikit-learn`.

In [ ]:
np.random.seed(42)
n_samples = 200
n_features = 3

X_multi = np.random.randn(n_samples, n_features)
true_coefs = np.array([1.5, -2.0, 3.0])
true_intercept_multi = 0.5
y_multi = true_intercept_multi + X_multi @ true_coefs + np.random.normal(0, 1.0, n_samples)

print(f"X_multi shape: {X_multi.shape}")
print(f"y_multi shape: {y_multi.shape}")

In [ ]:
# Normal equation: beta = (X^T X)^{-1} X^T y
# Add intercept column of ones
X_bias = np.column_stack([np.ones(n_samples), X_multi])
beta_multi = np.linalg.inv(X_bias.T @ X_bias) @ X_bias.T @ y_multi

print(f"Normal equation intercept: {beta_multi[0]:.4f}")
print(f"Normal equation coefs:     {beta_multi[1:]}")
print(f"True intercept:            {true_intercept_multi}")
print(f"True coefs:                {true_coefs}")

In [ ]:
reg_multi = LinearRegression()
reg_multi.fit(X_multi, y_multi)

print(f"sklearn intercept: {reg_multi.intercept_:.4f}")
print(f"sklearn coefs:     {reg_multi.coef_}")

y_hat_multi = reg_multi.predict(X_multi)
print(f"RMSE: {np.sqrt(mean_squared_error(y_multi, y_hat_multi)):.4f}")
print(f"R²:   {r2_score(y_multi, y_hat_multi):.4f}")

## 5. Train/test split and evaluation

Evaluate generalization performance on held-out data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

reg_train = LinearRegression()
reg_train.fit(X_train, y_train)

y_train_pred = reg_train.predict(X_train)
y_test_pred = reg_train.predict(X_test)

print("Train:")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
print(f"  R²:   {r2_score(y_train, y_train_pred):.4f}")

print("Test:")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")
print(f"  R²:   {r2_score(y_test, y_test_pred):.4f}")

## 6. Residual diagnostics

Plot residuals vs. fitted values to check for heteroscedasticity and nonlinearity.

In [ ]:
residuals = y_test - y_test_pred

plt.scatter(y_test_pred, residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--", linewidth=1)
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted (Test Set)")
plt.grid(True, alpha=0.3)
plt.show()

## 7. Regularization preview (Ridge / LASSO)

Standardize features before applying penalties.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

ridge = Ridge(alpha=1.0).fit(X_train_s, y_train)
lasso = Lasso(alpha=0.1, max_iter=10000).fit(X_train_s, y_train)

print("Ridge:")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test, ridge.predict(X_test_s))):.4f}")
print(f"  Test R²:   {r2_score(y_test, ridge.predict(X_test_s)):.4f}")
print(f"  Coefs:     {ridge.coef_}")

print("LASSO:")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test, lasso.predict(X_test_s))):.4f}")
print(f"  Test R²:   {r2_score(y_test, lasso.predict(X_test_s)):.4f}")
print(f"  Coefs:     {lasso.coef_}")

## Exercises

1. Vary the noise level in Section 1 and observe how RMSE and R² change.
2. Add a fourth irrelevant predictor to Section 4 and compare OLS vs. LASSO coefficient magnitudes.
3. Implement gradient descent for simple linear regression and compare to the closed-form solution.
4. Plot the learning curve (train vs. test error as training set size grows).